In [1]:
program double_pendulum_simulation
    implicit none
    integer, parameter :: n_steps = 600
    real(8), parameter :: dt = 0.02_8
    real(8), parameter :: g = 9.81_8
    real(8), parameter :: L1 = 1.0_8, L2 = 1.0_8
    real(8), parameter :: m1 = 1.0_8, m2 = 1.0_8
    real(8), parameter :: pi = 3.14159265358979_8

    real(8) :: theta1, omega1, theta2, omega2
    real(8) :: k1(4), k2(4), k3(4), k4(4)
    real(8) :: state(4), deriv(4)
    real(8) :: x1, y1, x2, y2
    integer :: step, unit_num
    character(len=100) :: filename

    ! Initial conditions: both arms displaced, released from rest
    theta1 = pi / 2.0_8
    omega1 = 0.0_8
    theta2 = pi / 2.0_8 + 0.1_8
    omega2 = 0.0_8

    call execute_command_line("mkdir -p double_pendulum_frames")

    do step = 1, n_steps
        x1 = L1 * sin(theta1)
        y1 = -L1 * cos(theta1)
        x2 = x1 + L2 * sin(theta2)
        y2 = y1 - L2 * cos(theta2)

        write(filename, '(A, I0, A)') &
            "double_pendulum_frames/f", step, ".dat"
        open(newunit=unit_num, file=trim(filename), status='replace')
        write(unit_num, '(F10.5, 1X, F10.5)') 0.0_8, 0.0_8
        write(unit_num, '(F10.5, 1X, F10.5)') x1, y1
        write(unit_num, '(F10.5, 1X, F10.5)') x2, y2
        close(unit_num)

        state = [theta1, omega1, theta2, omega2]

        call double_pendulum_deriv(state, deriv, g, L1, L2, m1, m2)
        k1 = deriv

        call double_pendulum_deriv(state + 0.5_8*dt*k1, deriv, g, L1, L2, m1, m2)
        k2 = deriv

        call double_pendulum_deriv(state + 0.5_8*dt*k2, deriv, g, L1, L2, m1, m2)
        k3 = deriv

        call double_pendulum_deriv(state + dt*k3, deriv, g, L1, L2, m1, m2)
        k4 = deriv

        state = state + (dt / 6.0_8) * (k1 + 2.0_8*k2 + 2.0_8*k3 + k4)

        theta1 = state(1)
        omega1 = state(2)
        theta2 = state(3)
        omega2 = state(4)
    end do

    print *, "Double pendulum frames written to double_pendulum_frames/"

contains

    subroutine double_pendulum_deriv(s, d, g, L1, L2, m1, m2)
        real(8), intent(in) :: s(4)
        real(8), intent(out) :: d(4)
        real(8), intent(in) :: g, L1, L2, m1, m2
        real(8) :: th1, w1, th2, w2
        real(8) :: delta, denom1, denom2

        th1 = s(1)
        w1  = s(2)
        th2 = s(3)
        w2  = s(4)

        delta = th2 - th1

        denom1 = (m1 + m2) * L1 - m2 * L1 * cos(delta) * cos(delta)
        denom2 = (L2 / L1) * denom1

        d(1) = w1
        d(2) = (m2*L1*w1*w1*sin(delta)*cos(delta) + &
                m2*g*sin(th2)*cos(delta) + &
                m2*L2*w2*w2*sin(delta) - &
                (m1+m2)*g*sin(th1)) / denom1

        d(3) = w2
        d(4) = (-m2*L2*w2*w2*sin(delta)*cos(delta) + &
                (m1+m2)*g*sin(th1)*cos(delta) - &
                (m1+m2)*L1*w1*w1*sin(delta) - &
                (m1+m2)*g*sin(th2)) / denom2
    end subroutine double_pendulum_deriv

end program double_pendulum_simulation

 Double pendulum frames written to double_pendulum_frames/


In [2]:
program generate_double_pendulum_animation
    implicit none
    integer :: unit_num, i
    integer, parameter :: n_steps = 600
    character(len=120) :: line

    open(newunit=unit_num, file='double_pendulum_animate.gp', status='replace')
    write(unit_num, '(A)') "set terminal gif animate delay 2 size 500,500"
    write(unit_num, '(A)') "set output 'double_pendulum.gif'"
    write(unit_num, '(A)') "set xrange [-2.2:2.2]"
    write(unit_num, '(A)') "set yrange [-2.2:2.2]"
    write(unit_num, '(A)') "unset key"
    write(unit_num, '(A)') "set size square"

    do i = 1, n_steps
        write(line, '(A, I0, A)') &
            "plot 'double_pendulum_frames/f", i, &
            ".dat' with linespoints lw 2 pt 7 ps 2"
        write(unit_num, '(A)') trim(line)
    end do

    close(unit_num)

    call execute_command_line("gnuplot double_pendulum_animate.gp")

    print *, "Animated GIF saved to double_pendulum.gif"
end program generate_double_pendulum_animation

600 frames in animation sequence


 Animated GIF saved to double_pendulum.gif


![Double pendulum animation](double_pendulum.gif)